<a href="https://colab.research.google.com/github/k242565-art/flyrank-ml-internship/blob/main/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/k242565-art/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*
Finding 1 – The Anatomy of Growing Content

The paper reports that growing content is longer (3.2K vs 2.3K words), younger (184 vs 230 days), and has slightly better average position than declining content. The paper notes that the comparison is observational and based on large sample sizes.
The finding shows a relationship between content growth, word count, and age. However, I would ask whether content length itself is driving growth, or whether longer content tends to belong to topics with higher search demand. A useful follow-up analysis would control for search volume and content intent to determine whether the observed relationship remains after accounting for those factors.

I would also ask whether the comparison was validated across different client groups. If the same brands contribute most of the growing content, the observed relationship may partially reflect brand-specific effects rather than a general content pattern.

Finding 2 – The Content Performance Curve

The paper reports that content performance peaks around 61–90 days, declines after 270 days, and that older content can recover when refreshed. The paper explicitly notes that the recovery of older pages is not evidence that age alone improves performance.
The paper observes a performance rebound for content older than 365 days. My primary question would be how many of those pages were actively refreshed versus left unchanged. If refreshed pages are driving the rebound, then the result reflects the effect of updates rather than the effect of age itself.

I would also ask whether the analysis tracks the same pages over time or compares different groups of pages at different ages. A longitudinal analysis following the same content throughout its lifecycle would provide stronger evidence for the proposed content lifecycle pattern.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*
The grouped validation score is considered a more realistic estimate because it evaluates performance on clients that were not seen during training. Any decrease in performance compared with the random split suggests that some patterns may be client-specific rather than broadly generalizable.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
print(os.getcwd())
import os

for root, dirs, files in os.walk("/content"):
    for f in files:
        if f == "content_refresh_anonymized.csv":
            print(os.path.join(root, f))
%cd /content
!git clone https://github.com/k242565-art/flyrank-ml-internship.git
print("X" in globals())
print("y" in globals())
print("df" in globals())
import os
print(os.getcwd())
!find /content -name "content_refresh_anonymized.csv"
import pandas as pd

df = pd.read_csv(
    "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"
)

print(df.shape)
df.head()
df["refresh_score"] = (
    df["days_since_last_update"] * 0.7
    +
    (-df["trend_pct"].clip(upper=0)) * 0.3
)

threshold = df["refresh_score"].quantile(0.75)

df["needs_refresh"] = (
    df["refresh_score"] >= threshold
).astype(int)

y = df["needs_refresh"]

print(y.value_counts())
features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

categorical = [
    "content_type",
    "main_intent",
    "competition_level",
    "provider_used",
    "model_used"
]

X = pd.get_dummies(
    df[features + categorical],
    drop_first=True
)
X = X.fillna(
    X.median(numeric_only=True)
)

X = X.fillna(0)

print(X.isnull().sum().sum())
print("df:", df.shape)
print("X:", X.shape)
print("y:", y.shape)
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        X,
        y,
        groups=df["client_id"]
    )
)
X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

/content
/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv
/content
fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.
True
True
True
/content
/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv
(30000, 44)
needs_refresh
0    23339
1     6661
Name: count, dtype: int64
0
df: (30000, 46)
X: (30000, 31)
y: (30000,)


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*
The model intentionally excluded refresh_score and needs_refresh from the feature set because they directly contribute to the target definition. Including either variable would create leakage and artificially inflate model performance.

All remaining features are historical observations or metadata that would be available at the time a refresh recommendation is generated.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*
The model accurately identifies content that needs refreshing.Within this dataset, the model demonstrated useful directional performance for identifying content that may benefit from editorial review.
Refreshing content improves SEO performance.The observed portfolio patterns suggest that content refresh activity may be associated with stronger performance, although this notebook does not establish a direct causal relationship.
The model can predict future content success.The model provides decision-support signals based on historical observations and should not be interpreted as a guarantee of future performance.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.